# Tutorial 03: ML/MM molecular dynamics in OpenMM

FieldMACE as an `openmm.PythonForce`, with any force field for the rest. Part 1 needs the model from tutorial 01,
part 2 the model and `work/` files from tutorial 02. Details: [README](README.md).

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")

## Part 1: ground state – NVT, then NVE

`create_fieldmace_mm_system(mm_system, qm_atoms, model, topology=..., settings=..., **prepare_kwargs)` removes
QM–QM bonded terms, zeroes the QM charges (QM–QM and QM–MM Coulomb come from FieldMACE), keeps QM–MM
Lennard-Jones, places link atoms and adds the model. `setup.json` from the training data guarantees the same
embedding.

In [ ]:
run(f"python 01_mlmm_md.py --nvt-ps {0.01 if QUICK else 2} --nve-ps {0.01 if QUICK else 1}")

In [ ]:
import matplotlib.pyplot as plt
d = np.loadtxt("results/mlmm_nve.csv", delimiter=",")
plt.figure(figsize=(5, 3))
for k, n in ((1, "E_pot"), (2, "E_kin"), (3, "E_tot")):
    plt.plot(d[:, 0], d[:, k] - d[0, k], label=n)
plt.xlabel("time (ps)"); plt.ylabel("energy - initial (kJ/mol)"); plt.legend(); plt.title("NVE"); plt.show()

## Part 2: excited states

The active state is the context parameter `fieldmace_state`; `FieldMACEStateReporter` writes all states'
energies. The script runs on S1, switches to S0 and rescales the velocities.

In [ ]:
if os.path.exists("../02_excited_states/models/ch2s_es.model"):
    run(f"python 02_multistate_md.py --ps-per-segment {0.005 if QUICK else 0.25}")
    display(Image("results/multistate.png"))
else:
    print("train the model of tutorial 02 first")

## Part 3: active learning

ML/MM MD with one model, a committee evaluates the frames, the most uncertain ones are written for labelling.

In [ ]:
run(f"python 03_active_learning.py --n-frames {4 if QUICK else 40} --every {5 if QUICK else 100} --n-select {2 if QUICK else 10}")